# Split Region / Country / City into proper columns

`Master_List.csv`'s `Region` column currently conflates three different things: a true multi-country region ("Central Asia"), a country name repeated as a region ("Indonesia", "Malaysia", "Singapore", "Sri Lanka"), and a metro-area label ("New York / New Jersey"). There's no real `City` column at all — `HQ / Primary Geography` holds a mix of city names, city+country pairs, and (for a few rows) no city at all.

This notebook:
1. Derives a proper `Region` (South Asia, North America, Southeast Asia, Central Asia, ...) from `Country`, via `data/region_mapping.csv` — a small, human-editable lookup table.
2. Derives `City` from `HQ / Primary Geography`, filtering out any token that's just restating the country or region (handles "Kuala Lumpur / Malaysia" → "Kuala Lumpur", "New York / New Jersey" → kept as-is since neither token is the country/region name, "Kazakhstan / Central Asia" → no real city disclosed → blank + flagged).
3. Leaves `Country` untouched (out of scope here) — including the one compound value ("Kazakhstan / Singapore"), which is handled as a special case for region/city lookup purposes only.

In [1]:
import shutil
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")
MASTER_PATH = DATA_DIR / "Master_List.csv"
REGION_MAPPING_PATH = DATA_DIR / "region_mapping.csv"

CITY_STATE_EXCEPTIONS = {"singapore"}  # country name doubles as its own primary city


def read_master(path):
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp1252")


master = read_master(MASTER_PATH)
region_mapping = pd.read_csv(REGION_MAPPING_PATH)
country_to_region = dict(zip(region_mapping["Country"], region_mapping["Region"]))

print(master.shape)
print(region_mapping)

(148, 32)
         Country          Region
0     Kazakhstan    Central Asia
1     Uzbekistan    Central Asia
2      Indonesia  Southeast Asia
3       Malaysia  Southeast Asia
4      Singapore  Southeast Asia
5      Sri Lanka      South Asia
6  United States   North America


## Derive proper `Region` from `Country`

For a compound `Country` value ("Kazakhstan / Singapore"), use the first-listed country as primary for region lookup — flagged for visibility, `Country` itself is left untouched.

In [2]:
def primary_country(country_value):
    return str(country_value).split("/")[0].strip()


master["_primary_country"] = master["Country"].apply(primary_country)

unmapped = sorted(set(master["_primary_country"]) - set(country_to_region))
if unmapped:
    print("WARNING: countries with no region_mapping.csv entry (will get blank Region):")
    for c in unmapped:
        print("  ", c)
else:
    print("all countries covered by region_mapping.csv")

compound_countries = master[master["Country"].str.contains("/", na=False)]
print(f"\n{len(compound_countries)} compound Country value(s) (using first token for region lookup):")
print(compound_countries[["Organisation", "Country", "_primary_country"]].to_string())

old_region = master["Region"].copy()
master["Region"] = master["_primary_country"].map(country_to_region)

print("\nold Region -> new Region (distinct pairs):")
print(pd.DataFrame({"old": old_region, "new": master["Region"]}).drop_duplicates().to_string())

all countries covered by region_mapping.csv

1 compound Country value(s) (using first token for region lookup):
                     Organisation                 Country _primary_country
12  Kusto Group / Yerkin Tatishev  Kazakhstan / Singapore       Kazakhstan

old Region -> new Region (distinct pairs):
                      old             new
0            Central Asia    Central Asia
18              Indonesia  Southeast Asia
37               Malaysia  Southeast Asia
57  New York / New Jersey   North America
77              Singapore  Southeast Asia
95              Sri Lanka      South Asia


## Derive `City` from `HQ / Primary Geography`

For each row: split on `/`, drop any token that *contains* the country name(s) or the new region name (substring match — catches messy cases like "Kazakhstan operations"). What's left is the city:
- One or more tokens survive → City = those tokens joined (preserves multi-office disclosures like "Almaty / Astana" instead of arbitrarily picking one).
- Nothing survives → the HQ value was just the country/region restated with no real city. Blank + flagged, **except** for city-states (Singapore) where the country name legitimately *is* the primary city.

In [3]:
def extract_city(hq_value, country_value, region_value):
    if pd.isna(hq_value):
        return None

    exclude_terms = [c.strip().lower() for c in str(country_value).split("/")]
    if pd.notna(region_value):
        exclude_terms.append(str(region_value).strip().lower())

    tokens = [t.strip() for t in str(hq_value).split("/") if t.strip()]
    candidates = [
        t for t in tokens
        if not any(term in t.lower() for term in exclude_terms)
    ]

    if candidates:
        return " / ".join(candidates)

    primary = primary_country(country_value).lower()
    if primary in CITY_STATE_EXCEPTIONS:
        return primary_country(country_value)

    return None  # no real city disclosed


master["City"] = master.apply(
    lambda r: extract_city(r["HQ / Primary Geography"], r["Country"], r["Region"]), axis=1
)

blank_city = master[master["City"].isna()]
print(f"{len(blank_city)} rows with no city extracted (flag for manual review):")
print(blank_city[["Organisation", "Country", "Region", "HQ / Primary Geography"]].to_string())

multi_city = master[master["City"].str.contains("/", na=False)]
print(f"\n{len(multi_city)} rows with multiple city tokens kept (review if a single primary city is preferred):")
print(multi_city[["Organisation", "HQ / Primary Geography", "City"]].drop_duplicates().to_string())

6 rows with no city extracted (flag for manual review):
                                       Organisation     Country        Region     HQ / Primary Geography
3                                       Resmi Group  Kazakhstan  Central Asia  Kazakhstan / Central Asia
6                                    Ordabasy Group  Kazakhstan  Central Asia                 Kazakhstan
7              EY Kazakhstan Family Office Services  Kazakhstan  Central Asia                 Kazakhstan
8                                Gryphon Kazakhstan  Kazakhstan  Central Asia                 Kazakhstan
10                      Visor Group / Visor Capital  Kazakhstan  Central Asia                 Kazakhstan
11  Almex Holding / Kulibayev family-linked holding  Kazakhstan  Central Asia                 Kazakhstan

30 rows with multiple city tokens kept (review if a single primary city is preferred):
                                                  Organisation      HQ / Primary Geography                        City
0 

## Reorder columns and write back to Master_List.csv (backup first)

In [4]:
master = master.drop(columns=["_primary_country"])

cols = list(master.columns)
cols.remove("City")
cols.insert(cols.index("Country") + 1, "City")
master = master[cols]

backup_path = MASTER_PATH.with_name(MASTER_PATH.stem + ".pre_region_cleanup.backup.csv")
if not backup_path.exists():
    shutil.copy(MASTER_PATH, backup_path)
    print("Backed up ->", backup_path)
else:
    print("Backup already exists, skipping ->", backup_path)

master.to_csv(MASTER_PATH, index=False)
print("Wrote", MASTER_PATH.resolve(), "rows:", len(master))
print(list(master.columns))

Backed up -> data\Master_List.pre_region_cleanup.backup.csv
Wrote C:\Users\USER\Desktop\TBP\tbp-dashboard\data\Master_List.csv rows: 148
['Region', 'Country', 'City', 'Organisation', 'Prospect Category', 'HQ / Primary Geography', 'Address / Office Location', 'Family / Founder / Strategic Nature', 'Known Sector Themes', 'TBP / Regional Corridor Relevance', 'Possible TBP Entry Point', 'Recommended Contact Route', 'Assigned Lead', 'Family Office Fit (20)', 'Permanent Capital (20)', 'Sector Alignment (20)', 'Governance Mindset (15)', 'Strategic Adjacency (15)', 'Engagement Readiness (10)', 'Total Score', 'Classification', 'Priority', 'Pipeline Stage', 'Scoring Status', 'Public Source URLs', 'Notes / Diligence Flags', 'Email address', 'Contact Email Status', 'Contact Email Source URLs', 'Contact Email Notes', 'Contact Enrichment Date', 'Source File', 'Alternate Names']


## Verify: reload from disk

In [5]:
reloaded = read_master(MASTER_PATH)
print(reloaded.shape)
print(reloaded[["Region", "Country", "City"]].head(10).to_string())
print()
print("Region value counts:")
print(reloaded["Region"].value_counts(dropna=False))
print()
print("rows with blank City:", reloaded["City"].isna().sum())

(148, 33)
         Region     Country             City
0  Central Asia  Kazakhstan  Almaty / Astana
1  Central Asia  Kazakhstan           Astana
2  Central Asia  Kazakhstan           Almaty
3  Central Asia  Kazakhstan              NaN
4  Central Asia  Kazakhstan           Astana
5  Central Asia  Kazakhstan           Almaty
6  Central Asia  Kazakhstan              NaN
7  Central Asia  Kazakhstan              NaN
8  Central Asia  Kazakhstan              NaN
9  Central Asia  Kazakhstan           Almaty

Region value counts:
Region
Southeast Asia    71
Central Asia      40
North America     20
South Asia        17
Name: count, dtype: int64

rows with blank City: 6
